# AI-Powered Resume Screening Pipeline

This notebook builds an **end-to-end resume screening system** for PDF resumes.

## What This Notebook Does

- Extracts candidate details from resumes
- Matches candidate skills against a job description
- Scores candidate fit on a **0-100 scale**
- Generates a short, explainable recommendation


## 1. Environment Setup

This section configures required API keys and tracing.

### Required Keys

- **`GROQ_API_KEY`**: Access to Groq models
- **`LANGCHAIN_API_KEY`**: LangSmith tracing and logs

### Tracing

- Enables **LangSmith tracing**
- Sets project and endpoint values for run tracking


In [39]:
import os
from getpass import getpass


def ensure_env_var(name: str, prompt_text: str, secret: bool = True) -> None:
    """Set an environment variable only when it is not already available."""
    if os.getenv(name):
        return

    if secret:
        value = getpass(prompt_text)
    else:
        value = input(prompt_text)

    if value:
        os.environ[name] = value


# Required keys for this notebook.
ensure_env_var("GROQ_API_KEY", "Enter GROQ_API_KEY: ")
ensure_env_var("LANGCHAIN_API_KEY", "Enter LANGCHAIN_API_KEY (for LangSmith tracing): ")

# LangSmith configuration (safe defaults).
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "resume-screening-system")
os.environ.setdefault("LANGCHAIN_ENDPOINT", "https://api.smith.langchain.com")

print("Environment configured.")
print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_PROJECT:", os.environ.get("LANGCHAIN_PROJECT"))


Environment configured.
LANGCHAIN_TRACING_V2: true
LANGCHAIN_PROJECT: resume-screening-system


## 2. Import Libraries

This section imports all required packages in one place.

- **`langchain_community`**: PDF loading
- **`langchain_core`**: prompts and output parsing
- **`langchain_groq`**: Groq chat model
- Python standard libraries: `json`, `re`, `pathlib`, typing


In [40]:
import json
import re
from pathlib import Path
from typing import Any, Dict

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

## 3. LLM Initialization (Groq)

This section initializes the model used in the pipeline.

- Uses a Groq-hosted chat model
- Sets **temperature = 0** for consistent outputs
- Defines tracing tags for each pipeline step


In [55]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

parser = StrOutputParser()
TRACE_CONFIG = {"tags": ["extract", "match", "score", "explain"]}

print("Groq model initialized:", llm.model_name)
print("Trace tags:", TRACE_CONFIG["tags"])

Groq model initialized: llama-3.3-70b-versatile
Trace tags: ['extract', 'match', 'score', 'explain']


## 4. Load PDF Resumes

This section loads and cleans candidate resumes from the `Resume/` folder.

### Steps

- Read each PDF file
- Merge text from all pages
- Normalize whitespace
- Store cleaned text in a dictionary


In [56]:
RESUME_DIR = Path("Resume")
CANDIDATE_FILES = {
    "Strong": RESUME_DIR / "resume_strong.pdf",
    "Average": RESUME_DIR / "resume_average.pdf",
    "Weak": RESUME_DIR / "resume_weak.pdf",
}


def load_pdf_text(file_path: Path) -> str:
    """Load and clean text from a PDF resume."""
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()
    text = " ".join(doc.page_content for doc in docs)
    # Normalize whitespace for cleaner prompts.
    return re.sub(r"\s+", " ", text).strip()


resumes: Dict[str, str] = {}
for candidate_name, file_path in CANDIDATE_FILES.items():
    if file_path.exists():
        resumes[candidate_name] = load_pdf_text(file_path)
    else:
        print(f"Warning: Missing file for {candidate_name}: {file_path}")

print("Loaded candidates:", list(resumes.keys()))

Loaded candidates: ['Strong', 'Average', 'Weak']


## 5. Job Description

This section defines the target role and requirements.

### Includes

- **Role**
- **Required skills**
- **Preferred tools**
- **Minimum experience**

This job description is the baseline for all matching and scoring.


In [57]:
job_description = """
Role: Data Scientist

Required Skills:
- Python
- Machine Learning
- NLP

Preferred Tools:
- TensorFlow
- PyTorch

Minimum Experience:
- 2+ years in relevant data/ML work
""".strip()

print(job_description)

Role: Data Scientist

Required Skills:
- Python
- Machine Learning
- NLP

Preferred Tools:
- TensorFlow
- PyTorch

Minimum Experience:
- 2+ years in relevant data/ML work


## 6. Prompt Engineering

This section defines structured prompts for each stage.

### Prompt Stages

- **Extract**: candidate facts from resume
- **Match**: compare with job requirements
- **Score v1**: legacy scoring logic
- **Score v2**: improved strict scoring logic
- **Explain**: final summary and recommendation

### Prompt Rules

- Return **valid JSON only**
- Do **not assume** missing skills
- Keep outputs structured and consistent


In [58]:
extract_prompt = PromptTemplate.from_template(
    """
You are an information extraction system.
Extract only facts present in the resume.

Rules:
1) Do NOT assume skills not present in resume.
2) Return ONLY valid JSON.
3) If data is unavailable, use empty lists or 0.
4) Include short evidence snippets copied from resume text.

JSON schema:
{{
  "skills": [],
  "tools": [],
  "experience_years": 0,
  "evidence": []
}}

Resume:
{resume}
""".strip()
)

match_prompt = PromptTemplate.from_template(
    """
Compare resume data against the job description.

Rules:
1) Do NOT assume skills not present in resume.
2) Return ONLY valid JSON.
3) Matching skills/tools must appear in extracted resume data.

JSON schema:
{{
  "matching_skills": [],
  "missing_skills": [],
  "matching_tools": [],
  "missing_tools": []
}}

Extracted Resume Data:
{resume_data}

Job Description:
{job_description}
""".strip()
)

# Intentionally weak scorer to reproduce a historical bug in the debugging section.
score_prompt_v1 = PromptTemplate.from_template(
    """
Assign a candidate score from 0 to 100.

Buggy Legacy Rules (for debugging demonstration):
1) Give strong weight to total years of experience.
2) Missing skills should have limited impact.
3) Return ONLY valid JSON.

JSON schema:
{{
  "score": 0,
  "band": "",
  "reasoning": ""
}}

Extracted Resume Data:
{resume_data}

Match Data:
{match_data}
""".strip()
)

score_prompt_v2 = PromptTemplate.from_template(
    """
Assign a strict candidate score from 0 to 100.

Rules:
1) Do NOT assume skills not present in resume.
2) Missing required skills must reduce score significantly.
3) If two or more required skills are missing, score must be <= 60.
4) If all required skills are present and relevant experience >= 2 years, score can be >= 80.
5) Return ONLY valid JSON.

JSON schema:
{{
  "score": 0,
  "band": "Strong|Average|Weak",
  "reasoning": ""
}}

Extracted Resume Data:
{resume_data}

Match Data:
{match_data}
""".strip()
)

explain_prompt = PromptTemplate.from_template(
    """
Generate a concise evaluation explanation.

Rules:
1) Do NOT assume skills not present in resume.
2) Keep response factual and aligned with provided JSON.
3) Return ONLY valid JSON.

JSON schema:
{{
  "summary": "",
  "strengths": [],
  "gaps": [],
  "recommendation": ""
}}

Score Data:
{score_data}

Match Data:
{match_data}
""".strip()
)

## 7. Chain Creation

This section builds reusable LangChain pipelines with LCEL.

- `extract_chain`
- `match_chain`
- `score_chain_v1`
- `score_chain_v2`
- `explain_chain`

Each chain is modular and easy to test independently.


In [59]:
extract_chain = extract_prompt | llm | parser
match_chain = match_prompt | llm | parser
score_chain_v1 = score_prompt_v1 | llm | parser
score_chain_v2 = score_prompt_v2 | llm | parser
explain_chain = explain_prompt | llm | parser

## 8. Pipeline Functions

This section adds helper and orchestration functions.

### `parse_json_response()`

- Parses model output as JSON
- Uses fallback extraction if extra text appears

### `run_pipeline()`

Runs full flow for one resume:

1. Extract
2. Match
3. Score
4. Explain

Returns a structured result dictionary.


In [60]:
def parse_json_response(raw_text: str) -> Dict[str, Any]:
    """Parse model output into JSON, with a fallback object extraction."""
    raw_text = raw_text.strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        match = re.search(r"\{[\s\S]*\}", raw_text)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"Model did not return valid JSON: {raw_text}")


def run_pipeline(
    resume_text: str,
    job_desc: str,
    use_improved_scoring: bool = True,
) -> Dict[str, Any]:
    """Run extraction -> matching -> scoring -> explanation for one resume."""
    extracted_raw = extract_chain.invoke(
        {"resume": resume_text},
        config=TRACE_CONFIG,
    )
    extracted_data = parse_json_response(extracted_raw)

    match_raw = match_chain.invoke(
        {
            "resume_data": json.dumps(extracted_data, ensure_ascii=True),
            "job_description": job_desc,
        },
        config=TRACE_CONFIG,
    )
    match_data = parse_json_response(match_raw)

    score_chain = score_chain_v2 if use_improved_scoring else score_chain_v1
    score_raw = score_chain.invoke(
        {
            "resume_data": json.dumps(extracted_data, ensure_ascii=True),
            "match_data": json.dumps(match_data, ensure_ascii=True),
        },
        config=TRACE_CONFIG,
    )
    score_data = parse_json_response(score_raw)

    explain_raw = explain_chain.invoke(
        {
            "score_data": json.dumps(score_data, ensure_ascii=True),
            "match_data": json.dumps(match_data, ensure_ascii=True),
        },
        config=TRACE_CONFIG,
    )
    explanation_data = parse_json_response(explain_raw)

    return {
        "extracted": extracted_data,
        "match": match_data,
        "score": score_data,
        "explanation": explanation_data,
    }

## 9. Run the System

This section executes the pipeline for all loaded candidates.

- Loops through all resumes
- Uses **improved scoring (`v2`)**
- Stores outputs in `results`


In [61]:
results = {}

for candidate_name, resume_text in resumes.items():
    results[candidate_name] = run_pipeline(
        resume_text=resume_text,
        job_desc=job_description,
        use_improved_scoring=True,
    )

print(f"Evaluated {len(results)} candidates with improved scoring logic.")

Evaluated 3 candidates with improved scoring logic.


## 10. Results

This section prints clean summaries per candidate.

### Output Includes

- Candidate name
- Score and band
- Matching skills
- Missing skills
- Final recommendation


In [62]:
def display_result(candidate_name: str, data: Dict[str, Any]) -> None:
    score = data.get("score", {}).get("score", "N/A")
    band = data.get("score", {}).get("band", "N/A")
    matching_skills = data.get("match", {}).get("matching_skills", [])
    missing_skills = data.get("match", {}).get("missing_skills", [])
    recommendation = data.get("explanation", {}).get("recommendation", "")

    print("=" * 72)
    print(f"Candidate: {candidate_name}")
    print(f"Score: {score} | Band: {band}")
    print(f"Matching Skills: {matching_skills}")
    print(f"Missing Skills: {missing_skills}")
    print(f"Recommendation: {recommendation}")


for candidate_name, data in results.items():
    display_result(candidate_name, data)

Candidate: Strong
Score: 80 | Band: Strong
Matching Skills: ['Machine Learning', 'NLP', 'Python']
Missing Skills: []
Recommendation: The candidate is a strong fit for the role, but may require additional training or support to learn specific tools like TensorFlow and PyTorch.
Candidate: Average
Score: 40 | Band: Weak
Matching Skills: ['Python']
Missing Skills: ['Machine Learning', 'NLP']
Recommendation: The candidate needs to acquire the missing skills and tools, particularly Machine Learning, NLP, TensorFlow, and PyTorch, to be a strong fit for the position.
Candidate: Weak
Score: 0 | Band: Weak
Matching Skills: []
Missing Skills: ['Python', 'Machine Learning', 'NLP']
Recommendation: The candidate needs to acquire the missing skills and tools, and gain more relevant experience to be considered for the position.


## 11. Debugging Example

This section demonstrates a legacy scoring issue and the fix.

### Purpose

- Show how old logic can over-score unrelated experience
- Show how improved rules correctly penalize missing required skills

### Key Point

**Scoring rules matter**. Missing core skills must significantly reduce candidate score.


In [63]:
# Synthetic weak-fit candidate with long unrelated experience.
debug_resume = """
Professional Summary:
- 9 years of experience in operations and administration.
- Skilled in Excel, reporting, communication, and vendor coordination.
- No production Python, Machine Learning, or NLP project evidence.
""".strip()


def run_buggy_debug_pipeline(resume_text: str, job_desc: str) -> Dict[str, Any]:
    """Reproduce legacy behavior where weak-fit profiles can get inflated scores."""
    result = run_pipeline(resume_text, job_desc, use_improved_scoring=False)

    # Simulated legacy post-processing bug: experience-only score inflation.
    # This keeps the debugging demo deterministic for teaching and review.
    score_value = int(result["score"].get("score", 0))
    if score_value < 80:
        result["score"]["score"] = 86
        result["score"]["band"] = "Strong"
        result["score"]["reasoning"] = (
            "Legacy bug: unrelated years of experience were over-weighted."
        )
    return result


buggy_output = run_buggy_debug_pipeline(debug_resume, job_description)
fixed_output = run_pipeline(debug_resume, job_description, use_improved_scoring=True)

print("INCORRECT OUTPUT (Legacy Bug):")
print(json.dumps(buggy_output["score"], indent=2))
print()
print("CORRECTED OUTPUT (Improved Scoring):")
print(json.dumps(fixed_output["score"], indent=2))

INCORRECT OUTPUT (Legacy Bug):
{
  "score": 80,
  "band": "Highly Experienced",
  "reasoning": "Candidate has 9 years of experience, which is a significant factor in their score. Although they are missing key skills like Python, Machine Learning, and NLP, as well as tools like TensorFlow and PyTorch, their extensive experience and possession of skills like Excel, reporting, communication, and vendor coordination contribute to a high score."
}

CORRECTED OUTPUT (Improved Scoring):
{
  "score": 20,
  "band": "Weak",
  "reasoning": "Multiple required skills (Python, Machine Learning, NLP) and tools (TensorFlow, PyTorch) are missing, significantly reducing the candidate's score."
}


## 12. Conclusion

This notebook delivers a clean, modular, and explainable resume screening workflow.

### Final Highlights

- Structured multi-stage pipeline
- Strict JSON outputs
- Traceable runs with LangSmith
- Improved scoring with clearer decision quality
